# Test State Injection Experiment

Tests the State Injection protocols (corner and middle) for Rotated Surface Code across different distances and injected states (|0⟩ and |+⟩).

In [ ]:
import sys
import os
import numpy as np

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from experiments.state_injection import StateInjectionExperiment

## 1. Build circuits (no noise)

Build injection circuits for different Distance, Protocol (corner/middle), and inject_state (Z->|0⟩, X->|+⟩).

In [ ]:
results = []
for d in [3, 5]:
    for protocol in ['corner', 'middle']:
        for inject_state in ['Z', 'X']:
            exp = StateInjectionExperiment(
                distance=d,
                rounds=2,
                injection_protocol=protocol,
                inject_state=inject_state,
                noise_params=None,
            )
            circ = exp.build()
            results.append({
                'distance': d,
                'protocol': protocol,
                'inject_state': inject_state,
                'circuit': circ,
                'num_ops': len(circ),
            })
            print(f'd={d} {protocol} {inject_state}: {len(circ)} ops')

## 2. Verify correctness by sampling (noiseless)

With zero noise, the circuit is deterministic. Sample multiple shots and check that the logical observable matches the injected state.

In [ ]:
n_shots = 100
all_pass = True
for r in results:
    circ = r['circuit']
    sampler = circ.compile_detector_sampler()
    dets, obs = sampler.sample(shots=n_shots, separate_observables=True)
    # obs shape: (n_shots, n_observables). For single logical, obs[:, 0]
    # Ideal |0⟩ (Z) and |+⟩ (X) both yield observable 0 in noiseless case
    if not np.all(obs == 0):
        all_pass = False
        print(f"FAIL d={r['distance']} {r['protocol']} {r['inject_state']}: obs={obs[:, 0][:5]}...")
    else:
        print(f"OK d={r['distance']} {r['protocol']} {r['inject_state']}: {n_shots} shots, obs=0 (correct)")

print("\n" + ("All tests passed!" if all_pass else "Some tests failed."))

## 3. Circuit diagram (example: d=3, corner, Z)

In [ ]:
exp = StateInjectionExperiment(distance=3, rounds=2, injection_protocol='corner', inject_state='Z')
circ = exp.build()
circ.without_noise().diagram('detslice-with-ops-svg')